# Model Training

Train ML models for price prediction.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import joblib

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("="*70)
print("MODEL TRAINING ON KAGGLE DATA - LSTM")
print("="*70)
print(f"Training Period: {TRAIN_START} to {TRAIN_END}")
print(f"Test Period: {TEST_START} to {TEST_END}")
print("="*70)

MODEL TRAINING ON KAGGLE DATA - LSTM
Training Period: 2015-01-01 to 2023-12-31
Test Period: 2024-01-01 to 2024-12-31


c:\Users\Sunay Bhattacharjee\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# Load and prepare data for first ticker
ticker = DEFAULT_TICKERS[0]
print(f"\nLoading {ticker}...")

raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)

print(f"Cleaned data shape: {cleaned_data.shape}")
print(f"Date range: {cleaned_data.index[0]} to {cleaned_data.index[-1]}")

# Split by dates
train_data, test_data = split_data_by_date(cleaned_data)
print(f"\nTrain data: {train_data.shape}")
print(f"Test data: {test_data.shape}")


Loading NIFTY BANK...
2025-12-22 17:48:57 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-22 17:48:58 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-22 17:48:58 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 

## Check Available Data Intervals

In [3]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Add features (LSTM / ML friendly, no leakage)
def add_basic_features(data):
    df = data.copy()

    # -----------------------------
    # Returns
    # -----------------------------
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # -----------------------------
    # Moving averages (normalized)
    # -----------------------------
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # -----------------------------
    # Price action
    # -----------------------------
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]

    # -----------------------------
    # Volatility
    # -----------------------------
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()

    # -----------------------------
    # RSI (normalized 0–1)
    # -----------------------------
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # -----------------------------
    # Volume (safe for indices)
    # -----------------------------
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # -----------------------------
    # Target
    # -----------------------------
    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    # -----------------------------
    # Cleanup (NO backfill → no leakage)
    # -----------------------------
    df = df.dropna()

    return df


# -------------------------------------------------
# Apply feature engineering
# -------------------------------------------------
train_with_features = add_basic_features(train_data)
test_with_features = add_basic_features(test_data)

print(f"Train data shape before features: {train_data.shape}")
print(f"Test data shape before features : {test_data.shape}")

print(f"\nTrain with features: {train_with_features.shape}")
print(f"Test with features : {test_with_features.shape}")

if not train_with_features.empty:
    feature_cols = [c for c in train_with_features.columns if c != "target"]
    print(f"\nFeatures used ({len(feature_cols)}):")
    print(feature_cols)
else:
    print("WARNING: No training data after feature engineering!")

if test_with_features.empty:
    print("WARNING: No test data after feature engineering!")


Train data shape before features: (791657, 5)
Test data shape before features : (80907, 5)

Train with features: (791543, 17)
Test with features : (80848, 17)

Features used (16):
['Open', 'High', 'Low', 'Close', 'Volume', 'returns', 'log_returns', 'trend_10', 'trend_20', 'trend_diff', 'range_pct', 'body_pct', 'volatility_10', 'vol_ratio', 'RSI', 'Volume_norm']


In [5]:
# Prepare X and y
feature_cols = [col for col in train_with_features.columns if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Target distribution (train): {y_train.value_counts().to_dict()}")

X_train shape: (791543, 11)
y_train shape: (791543,)
Target distribution (train): {0: 397081, 1: 394462}


In [6]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# -----------------------------
# Scale features (LSTM-friendly)
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled train shape: {X_train_scaled.shape}")
print(f"Scaled test shape : {X_test_scaled.shape}")

# -----------------------------
# Sequence length
# -----------------------------
sequence_length = 10


def create_sequences_np(data, labels, seq_length):
    """
    Create LSTM sequences with proper alignment.
    data   : np.ndarray (features)
    labels : np.ndarray (targets)
    """
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length:i])
        y.append(labels[i])
    return np.array(X), np.array(y)


# Convert labels to numpy
y_train_np = y_train.values
y_test_np = y_test.values

# -----------------------------
# Train sequences
# -----------------------------
X_train_seq, y_train_seq = create_sequences_np(
    X_train_scaled,
    y_train_np,
    sequence_length
)

# -----------------------------
# Test sequences (IMPORTANT FIX)
# Use last part of train as context
# -----------------------------
X_test_extended = np.vstack([
    X_train_scaled[-sequence_length:],
    X_test_scaled
])

y_test_extended = np.concatenate([
    y_train_np[-sequence_length:],
    y_test_np
])

X_test_seq, y_test_seq = create_sequences_np(
    X_test_extended,
    y_test_extended,
    sequence_length
)

print("\nLSTM sequence shapes:")
print(f"X_train_seq: {X_train_seq.shape}")
print(f"y_train_seq: {y_train_seq.shape}")
print(f"X_test_seq : {X_test_seq.shape}")
print(f"y_test_seq : {y_test_seq.shape}")


Scaled train shape: (791543, 11)
Scaled test shape : (80848, 11)

LSTM sequence shapes:
X_train_seq: (791533, 10, 11)
y_train_seq: (791533,)
X_test_seq : (80848, 10, 11)
y_test_seq : (80848,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("\nTraining Advanced LSTM model...")

# ===================================
# 1. CLASS WEIGHTS (handle imbalance)
# ===================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_seq),
    y=y_train_seq
)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

print(f"Class weights: {class_weight_dict}")
print(f"Train target distribution: {np.bincount(y_train_seq)}")

# ===================================
# 2. TEMPORAL TRAIN/VAL SPLIT
# ===================================
val_split = int(0.8 * len(X_train_seq))
X_tr, X_val = X_train_seq[:val_split], X_train_seq[val_split:]
y_tr, y_val = y_train_seq[:val_split], y_train_seq[val_split:]

# ===================================
# 3. ADVANCED LSTM ARCHITECTURE
# ===================================
lstm_model = Sequential([
    # First Bidirectional LSTM block
    Bidirectional(
        LSTM(128, activation='tanh', return_sequences=True, 
             kernel_regularizer='l2', recurrent_regularizer='l2'),
        input_shape=(sequence_length, X_train_seq.shape[2])
    ),
    Dropout(0.4),
    BatchNormalization(),
    
    # Second Bidirectional LSTM block
    Bidirectional(
        LSTM(64, activation='tanh', return_sequences=True,
             kernel_regularizer='l2', recurrent_regularizer='l2')
    ),
    Dropout(0.4),
    BatchNormalization(),
    
    # Third LSTM layer (without bidirectional for efficiency)
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.3),
    BatchNormalization(),
    
    # Dense layers
    Dense(64, activation='relu', kernel_regularizer='l2'),
    Dropout(0.3),
    Dense(32, activation='relu', kernel_regularizer='l2'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC']
)

print(lstm_model.summary())

# ===================================
# 4. ADVANCED CALLBACKS
# ===================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# ===================================
# 5. TRAINING WITH CLASS WEIGHTS
# ===================================
history = lstm_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ===================================
# 6. THRESHOLD OPTIMIZATION
# ===================================
y_train_prob = lstm_model.predict(X_train_seq, verbose=0).flatten()
y_test_prob = lstm_model.predict(X_test_seq, verbose=0).flatten()

# Find optimal threshold using F1 score
best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.3, 0.7, 0.05):
    y_train_pred_temp = (y_train_prob > threshold).astype(int)
    f1 = f1_score(y_train_seq, y_train_pred_temp, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nOptimal threshold: {best_threshold:.2f} (F1: {best_f1:.4f})")

# Apply optimal threshold
y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)

# ===================================
# 7. METRICS
# ===================================
train_accuracy = accuracy_score(y_train_seq, y_train_pred)
test_accuracy = accuracy_score(y_test_seq, y_test_pred)
train_auc = roc_auc_score(y_train_seq, y_train_prob)
test_auc = roc_auc_score(y_test_seq, y_test_prob)
train_f1 = f1_score(y_train_seq, y_train_pred)
test_f1 = f1_score(y_test_seq, y_test_pred)

print(f"\n{'='*50}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train AUC:      {train_auc:.4f}")
print(f"Train F1:       {train_f1:.4f}")
print(f"{'='*50}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test AUC:       {test_auc:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"{'='*50}")


Training LSTM model (improved)...


c:\Users\Sunay Bhattacharjee\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 64)         │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,289 (126.13 KB)

 Trainable params: 32,097 (125.38 KB)

 Non-trainable params: 192 (768.00 B)

None
Epoch 1/40
5271/9895 ━━━━━━━━━━━━━━━━━━━━ 28s 6ms/step - accuracy: 0.5064 - loss: 0.7240

KeyboardInterrupt: 

In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test_seq, y_test_pred))

print("\nConfusion Matrix (Test Set):")
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test_seq, y_test_pred))


Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.53      0.48      0.50     40550
           1       0.52      0.57      0.54     40356

    accuracy                           0.52     80906
   macro avg       0.52      0.52      0.52     80906
weighted avg       0.52      0.52      0.52     80906



## Improve Model Accuracy

Try these optimization strategies to increase accuracy:

In [ ]:
# Train improved LSTM models for all tickers
results = {}

print("\n" + "="*70)
print("TRAINING ADVANCED LSTM FOR ALL TICKERS")
print("="*70)

for ticker in DEFAULT_TICKERS:
    print(f"\nTraining {ticker}...", end=" ")
    
    try:
        # Load and prepare
        raw_data = load_kaggle_data(ticker)
        cleaned = clean_ohlcv_data(raw_data)
        train, test = split_data_by_date(cleaned)
        
        # Features
        train_f = add_basic_features(train)
        test_f = add_basic_features(test)
        
        if len(train_f) == 0 or len(test_f) == 0:
            print("SKIPPED (no data)")
            continue
        
        # Prepare X and y
        X_train_ticker = train_f[feature_cols]
        y_train_ticker = train_f['target'].values
        X_test_ticker = test_f[feature_cols]
        y_test_ticker = test_f['target'].values
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_ticker)
        X_test_scaled = scaler.transform(X_test_ticker)
        
        # Create sequences with extended test set (for context)
        X_train_seq_t, y_train_seq_t = create_sequences_np(X_train_scaled, y_train_ticker, sequence_length)
        
        X_test_extended_t = np.vstack([X_train_scaled[-sequence_length:], X_test_scaled])
        y_test_extended_t = np.concatenate([y_train_ticker[-sequence_length:], y_test_ticker])
        X_test_seq_t, y_test_seq_t = create_sequences_np(X_test_extended_t, y_test_extended_t, sequence_length)
        
        if len(X_train_seq_t) < 50 or len(X_test_seq_t) < 10:
            print("SKIPPED (insufficient sequences)")
            continue
        
        # Compute class weights
        cw = compute_class_weight('balanced', classes=np.unique(y_train_seq_t), y=y_train_seq_t)
        cw_dict = {i: cw[i] for i in range(len(cw))}
        
        # Temporal split
        val_split_t = int(0.8 * len(X_train_seq_t))
        X_tr_t, X_val_t = X_train_seq_t[:val_split_t], X_train_seq_t[val_split_t:]
        y_tr_t, y_val_t = y_train_seq_t[:val_split_t], y_train_seq_t[val_split_t:]
        
        # Build advanced LSTM
        model = Sequential([
            Bidirectional(
                LSTM(96, activation='tanh', return_sequences=True, 
                     kernel_regularizer='l2', recurrent_regularizer='l2'),
                input_shape=(sequence_length, X_train_seq_t.shape[2])
            ),
            Dropout(0.35),
            BatchNormalization(),
            
            Bidirectional(
                LSTM(48, activation='tanh', return_sequences=True,
                     kernel_regularizer='l2', recurrent_regularizer='l2')
            ),
            Dropout(0.35),
            BatchNormalization(),
            
            LSTM(24, activation='tanh'),
            Dropout(0.3),
            
            Dense(48, activation='relu', kernel_regularizer='l2'),
            Dropout(0.2),
            Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy', 'AUC']
        )
        
        # Callbacks
        es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)
        rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=0)
        
        # Train
        model.fit(
            X_tr_t, y_tr_t,
            validation_data=(X_val_t, y_val_t),
            epochs=50,
            batch_size=32,
            class_weight=cw_dict,
            callbacks=[es, rlr],
            verbose=0
        )
        
        # Threshold optimization
        y_train_prob_t = model.predict(X_train_seq_t, verbose=0).flatten()
        best_thresh = 0.5
        best_f1_t = 0
        for thresh in np.arange(0.3, 0.7, 0.05):
            y_pred_temp = (y_train_prob_t > thresh).astype(int)
            f1 = f1_score(y_train_seq_t, y_pred_temp, zero_division=0)
            if f1 > best_f1_t:
                best_f1_t = f1
                best_thresh = thresh
        
        # Evaluate on test
        y_test_prob_t = model.predict(X_test_seq_t, verbose=0).flatten()
        y_test_pred_t = (y_test_prob_t > best_thresh).astype(int)
        test_acc_t = accuracy_score(y_test_seq_t, y_test_pred_t)
        test_auc_t = roc_auc_score(y_test_seq_t, y_test_prob_t)
        
        results[ticker] = {
            'model': model,
            'scaler': scaler,
            'features': feature_cols,
            'sequence_length': sequence_length,
            'threshold': best_thresh,
            'train_samples': len(X_train_seq_t),
            'test_samples': len(X_test_seq_t),
            'test_accuracy': test_acc_t,
            'test_auc': test_auc_t
        }
        
        print(f"✅ Accuracy: {test_acc_t:.4f} | AUC: {test_auc_t:.4f} | Threshold: {best_thresh:.2f}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)[:60]}")

print("\n" + "="*70)
print(f"Successfully trained {len(results)} LSTM models")
print("="*70)

# Summary
if results:
    accs = [v['test_accuracy'] for v in results.values()]
    aucs = [v['test_auc'] for v in results.values()]
    print(f"\nAverage Accuracy: {np.mean(accs):.4f} (±{np.std(accs):.4f})")
    print(f"Average AUC:      {np.mean(aucs):.4f} (±{np.std(aucs):.4f})")

In [ ]:
# Option 2: Use GridSearchCV to find best XGBoost hyperparameters
best_xgb, best_cv_score = tune_xgboost_hyperparameters(X_train, y_train)

# Then train stacked model with tuned hyperparameters
model_tuned, tuned_accuracy = train_tuned_stacked_model(X_train, y_train, X_test, y_test)
y_pred_tuned = model_tuned.predict(X_test)

print(f"\n--- Accuracy Comparison ---")
print(f"Original Model:  {acc_original:.4f}")
print(f"Improved Model:  {acc_improved:.4f}")
print(f"Tuned Model:     {tuned_accuracy:.4f}")

# Evaluate tuned model
metrics_tuned = evaluate_classification(y_test, y_pred_tuned, model_name='Tuned Stacked (RF + XGB + GB)')

### Other Strategies to Improve Accuracy:

1. **Feature Engineering**: Create new features or remove irrelevant ones
2. **Data Preprocessing**: Better handling of outliers, normalization
3. **Class Imbalance**: Use SMOTE or adjust class weights further
4. **Ensemble**: Add more diverse base learners (SVM, Neural Networks)
5. **Target Engineering**: Ensure target variable is well-defined
6. **Feature Scaling**: Try different scaling methods